# Module 4: Open Quantum Systems, Dissipation & Trajectories
## Notebook 4.5: Quantum Trajectories & Homodyne Detection Unraveling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_4_open_systems_dissipation/06_quantum_trajectories_and_homodyne_measurement.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** **Open Quantum Systems, Dissipation & Trajectories**
  * **4.1** Density Matrix, Purity & Entanglement Entropy
  * **4.2** The Lindblad Master Equation & Dynamical Semigroups
  * **4.3** Cavity Decay, Photon Loss & Thermal Reservoirs
  * **4.4** The Monte Carlo Wavefunction (Quantum Jump) Method
  * **4.5 Quantum Trajectories & Homodyne Detection Unraveling** $\leftarrow$ *(You are here)*
  * **4.6** Input-Output Theory & Cavity Scattering
  * **4.7** Laser Theory, Micromaser & Trapping States

---

### Learning Objectives
1. **Unravelings of Open Quantum Systems:** Understand how the choice of environmental measurement (photo-detection vs. homodyne detection) selects different stochastic unravelings of the identical Lindblad master equation.
2. **Balanced Homodyne Detection:** Model the interference of cavity output with an intense classical local oscillator ($|\beta| \to \infty$) and derive the continuous homodyne photocurrent $J(t) dt = 2\sqrt{\kappa}\langle\hat{x}\rangle_t dt + dW(t)$.
3. **The Stochastic Schrödinger Equation (SSE):** Formulate the diffusive Itô stochastic differential equation governing conditional state evolution under continuous quadrature monitoring.
4. **Measurement Backaction & Conditional Squeezing:** Demonstrate how continuous monitoring purifies the system and generates conditional quadrature squeezing ($\Delta x^2 < 1/2$) below the vacuum limit.
5. **Diffusive Convergence to GKSL:** Numerically verify that the statistical ensemble average over noisy diffusive trajectories recovers the unconditional Lindblad master equation.



In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "lines.linewidth": 1.8,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

# Verify environment capabilities
print(f"NumPy version: {np.__version__}")
try:
    import qutip as qt
    print(f"QuTiP version: {qt.__version__} (Available for optional comparison)")
    HAS_QUTIP = True
except ImportError:
    print("QuTiP not installed; running pure, optimized native NumPy/SciPy engine.")
    HAS_QUTIP = False



---

## Theoretical Framework: Continuous Measurement & Diffusive Trajectories

### 1. From Quantum Jumps to Quantum State Diffusion
In Notebook 4.4, we studied the **discrete jump unraveling** corresponding to direct photodetector counting: each detected photon causes an instantaneous projective jump $\hat{a}|\psi\rangle$. 
However, in quantum optics and superconducting circuit QED, emitted radiation is frequently measured via **balanced homodyne detection**.

In a balanced homodyne setup:
1. The cavity output field $\hat{a}_{\text{out}}$ is mixed with an intense coherent **Local Oscillator (LO)** $|\beta e^{i\theta}\rangle$ on a 50:50 beam splitter.
2. The two output ports are monitored by photodetectors $D_1$ and $D_2$, producing photocurrents $I_1(t)$ and $I_2(t)$.
3. The difference photocurrent $I(t) = I_1(t) - I_2(t)$ is amplified.

In the limit of an asymptotically strong local oscillator ($|\beta| \to \infty$):
* The individual photon clicks become infinitely frequent ($\sim |\beta|^2$).
* The discrete Poisson process converges (via the Central Limit Theorem) to a continuous **diffusive Gaussian stochastic process** (Wiener process).

---

### 2. The Homodyne Photocurrent Record
For a local oscillator phase $\theta$, the balanced homodyne detection continuously measures the generalized optical quadrature:
$$\hat{x}_\theta = \frac{\hat{a} e^{-i\theta} + \hat{a}^\dagger e^{i\theta}}{2}$$

For $\theta = 0$, $\hat{x}_0 = \hat{x} = \frac{\hat{a} + \hat{a}^\dagger}{2}$ (position/amplitude quadrature).
For $\theta = \pi/2$, $\hat{x}_{\pi/2} = \hat{p} = \frac{\hat{a} - \hat{a}^\dagger}{2i}$ (momentum/phase quadrature).

The continuous differential photocurrent record $J(t) dt$ registered by the detector during interval $[t, t + dt]$ is given by:
$$J(t) dt = 2\sqrt{\kappa \eta} \, \langle\hat{x}_\theta\rangle_t \, dt + dW(t)$$
where:
* $\kappa$ is the cavity decay rate.
* $\eta \in (0, 1]$ is the quantum efficiency of the detector.
* $\langle\hat{x}_\theta\rangle_t = \langle\psi(t)|\hat{x}_\theta|\psi(t)\rangle$ is the conditional expectation value of the quadrature at time $t$.
* $dW(t)$ is a real-valued **Wiener increment** representing fundamental quantum measurement shot noise, satisfying:
  $$\mathbb{E}[dW(t)] = 0, \quad dW(t)^2 = dt \quad (\text{Itô rule})$$

---

### 3. The Diffusive Stochastic Schrödinger Equation (SSE)
Conditioned on measuring photocurrent increment $J(t) dt$, the quantum state $|\psi(t)\rangle$ updates via the **Stochastic Schrödinger Equation (SSE)** in Itô form. 
For an internal Hamiltonian $\hat{H}$ and jump operator $\hat{c} = \sqrt{\kappa}\hat{a}$ with local oscillator phase $\theta = 0$:

$$d|\psi\rangle = \left[ -i\hat{H}dt - \frac{\kappa}{2}\left(\hat{a}^\dagger\hat{a} - 2\langle\hat{x}\rangle_t\hat{a} + \langle\hat{x}\rangle_t^2\right)dt + \sqrt{\kappa}\left(\hat{a} - \langle\hat{x}\rangle_t\right)dW \right] |\psi(t)\rangle$$

#### Key Features of the Diffusive SSE:
1. **Norm Preservation:** Because $dW^2 = dt$, the Itô expansion of $d\langle\psi|\psi\rangle$ yields:
   $$d(\langle\psi|\psi\rangle) = \langle\psi|d\psi\rangle + \langle d\psi|\psi\rangle + \langle d\psi|d\psi\rangle = 0$$
   The state vector remains strictly normalized ($\langle\psi(t)|\psi(t)\rangle = 1$) at all times!
2. **Measurement Backaction (Innovation):** The term $\sqrt{\kappa}(\hat{a} - \langle\hat{x}\rangle_t)dW$ represents the measurement update. If a fluctuation $dW > 0$ occurs, the state is dynamically kicked in the direction of the eigenbasis of $\hat{x}$.
3. **Equivalence to GKSL:** Taking the stochastic ensemble expectation over all realizations of the Wiener noise ($\mathbb{E}[dW] = 0$):
   $$\mathbb{E}[d(|\psi\rangle\langle\psi|)] = \left( -i[\hat{H}, \hat{\rho}] + \kappa\mathcal{D}[\hat{a}]\hat{\rho} \right) dt$$
   The ensemble average of pure conditional trajectories $|\psi(t)\rangle$ recovers the exact unconditional Lindblad master equation!

---

### 4. Measurement Backaction and Conditional Squeezing
In an unconditional open system, photon loss heats or decoheres quantum states into mixed states ($\text{Tr}(\hat{\rho}^2) < 1$). 
However, under continuous homodyne monitoring:
* Every trajectory remains in a **pure state** ($\text{Tr}(\hat{\rho}^2) = 1$).
* The continuous extraction of information about $\hat{x}$ dynamically collapses quantum fluctuations along that quadrature.
* For a driven cavity or parametric oscillator, continuous homodyne monitoring generates **conditional squeezing**: the variance $\Delta x^2(t) = \langle\hat{x}^2\rangle_t - \langle\hat{x}\rangle_t^2$ drops below the standard quantum limit ($\Delta x^2 < 1/4$), even if the unmonitored state is completely classical!



In [ ]:
# Implementation: High-Precision Itô Stochastic Schrödinger Equation (SSE) Engine

class DiffusiveTrajectorySimulator:
    """
    Diffusive Quantum Trajectory Engine for Cavity Homodyne Measurement.
    
    Model: Cavity mode under coherent drive and homodyne quadrature detection:
        H_sys = Delta * a^dagger * a + i * E * (a^dagger - a)
        c = sqrt(kappa) * a
    """
    def __init__(self, n_cutoff=25, delta=0.0, drive=1.0, kappa=1.0, lo_phase=0.0, eta=1.0):
        self.dim = n_cutoff
        self.delta = float(delta)
        self.drive = float(drive)
        self.kappa = float(kappa)
        self.theta = float(lo_phase)
        self.eta = float(eta)  # Detection efficiency
        
        # Operators in Fock basis
        self.a = np.zeros((self.dim, self.dim), dtype=complex)
        for n in range(1, self.dim):
            self.a[n-1, n] = np.sqrt(n)
        self.a_dag = self.a.conj().T
        self.num_op = np.dot(self.a_dag, self.a)
        
        # Quadratures
        # x = (a + a^dagger)/2, p = (a - a^dagger)/(2i)
        self.x_op = 0.5 * (self.a + self.a_dag)
        self.p_op = -0.5j * (self.a - self.a_dag)
        
        # Monitored quadrature at angle theta: x_theta = (a e^-itheta + a^dagger e^itheta)/2
        self.x_theta_op = 0.5 * (self.a * np.exp(-1j * self.theta) + self.a_dag * np.exp(1j * self.theta))
        
        # System Hamiltonian: detuning + coherent driving
        self.H_sys = self.delta * self.num_op + 1j * self.drive * (self.a_dag - self.a)
        self.c_op = np.sqrt(self.kappa) * self.a

    def run_homodyne_trajectory(self, psi0, t_list, seed=None):
        """
        Integrate the diffusive Stochastic Schrödinger Equation (SSE) using Itô-Euler-Maruyama.
        
        Parameters:
            psi0: Initial normalized state vector (shape: (dim,))
            t_list: 1D array of time points (equally spaced)
            seed: RNG seed
            
        Returns:
            dict containing:
                't': time array
                'x': conditional expectation <x>(t)
                'p': conditional expectation <p>(t)
                'var_x': conditional variance (Delta x)^2(t)
                'n': conditional mean photon number <n>(t)
                'photocurrent': homodyne current record J(t)
        """
        rng = np.random.default_rng(seed)
        n_steps = len(t_list)
        dt = t_list[1] - t_list[0]
        sqrt_dt = np.sqrt(dt)
        
        psi = np.array(psi0, dtype=complex) / la.norm(psi0)
        
        x_rec = np.zeros(n_steps, dtype=float)
        p_rec = np.zeros(n_steps, dtype=float)
        var_x_rec = np.zeros(n_steps, dtype=float)
        n_rec = np.zeros(n_steps, dtype=float)
        current_rec = np.zeros(n_steps, dtype=float)
        
        def get_exp(op, state):
            return float(np.real(np.dot(state.conj().T, np.dot(op, state))))
        
        x_rec[0] = get_exp(self.x_op, psi)
        p_rec[0] = get_exp(self.p_op, psi)
        var_x_rec[0] = get_exp(np.dot(self.x_op, self.x_op), psi) - x_rec[0]**2
        n_rec[0] = get_exp(self.num_op, psi)
        current_rec[0] = 2.0 * np.sqrt(self.kappa * self.eta) * x_rec[0]
        
        a_theta = self.a * np.exp(-1j * self.theta)
        
        for idx in range(1, n_steps):
            # Draw Wiener increment: dW ~ N(0, dt)
            dW = rng.normal(0.0, sqrt_dt)
            
            # Current expectation of monitored quadrature
            x_meas = get_exp(self.x_theta_op, psi)
            
            # Record measured homodyne photocurrent: J(t) dt = 2*sqrt(kappa*eta)*<x_theta> dt + dW
            current_rec[idx] = 2.0 * np.sqrt(self.kappa * self.eta) * x_meas + (dW / dt)
            
            # Drift term acting on psi:
            drift = -1j * np.dot(self.H_sys, psi) * dt
            # Dissipative drift
            term1 = np.dot(self.num_op, psi)
            term2 = 2.0 * x_meas * np.dot(a_theta, psi)
            term3 = (x_meas**2) * psi
            drift -= 0.5 * self.kappa * (term1 - term2 + term3) * dt
            
            # Diffusion term acting on psi:
            diff_term = (np.dot(a_theta, psi) - x_meas * psi)
            diffusion = np.sqrt(self.kappa * self.eta) * diff_term * dW
            
            # Update state vector
            psi_new = psi + drift + diffusion
            
            # Normalize to guarantee exact numerical stability
            norm_val = la.norm(psi_new)
            psi = psi_new / norm_val
            
            # Record observables
            x_val = get_exp(self.x_op, psi)
            p_val = get_exp(self.p_op, psi)
            x_sq = get_exp(np.dot(self.x_op, self.x_op), psi)
            
            x_rec[idx] = x_val
            p_rec[idx] = p_val
            var_x_rec[idx] = x_sq - x_val**2
            n_rec[idx] = get_exp(self.num_op, psi)
            
        return {
            "t": t_list,
            "x": x_rec,
            "p": p_rec,
            "var_x": var_x_rec,
            "n": n_rec,
            "current": current_rec
        }

    def run_ensemble(self, psi0, t_list, n_trajectories=50, base_seed=100):
        """
        Run ensemble of homodyne trajectories to benchmark against Lindblad master equation.
        """
        n_steps = len(t_list)
        x_all = np.zeros((n_trajectories, n_steps))
        n_all = np.zeros((n_trajectories, n_steps))
        var_x_all = np.zeros((n_trajectories, n_steps))
        
        for k in range(n_trajectories):
            seed = base_seed + k if base_seed is not None else None
            res = self.run_homodyne_trajectory(psi0, t_list, seed=seed)
            x_all[k, :] = res["x"]
            n_all[k, :] = res["n"]
            var_x_all[k, :] = res["var_x"]
            
        return {
            "t": t_list,
            "mean_x": np.mean(x_all, axis=0),
            "sem_x": np.std(x_all, axis=0) / np.sqrt(n_trajectories),
            "mean_n": np.mean(n_all, axis=0),
            "sem_n": np.std(n_all, axis=0) / np.sqrt(n_trajectories),
            "mean_var_x": np.mean(var_x_all, axis=0)
        }

    def solve_lindblad_exact(self, psi0, t_list):
        """
        Solve unconditional Lindblad master equation: d_rho/dt = -i[H, rho] + kappa * D[a] rho
        Propagates state with precomputed single-step propagator exp(L * dt) for maximum speed.
        """
        rho0 = np.outer(psi0, psi0.conj())
        d = self.dim
        Id = np.eye(d, dtype=complex)
        dt = t_list[1] - t_list[0]
        
        # Liouvillian superoperator
        L_super = -1j * (np.kron(self.H_sys, Id) - np.kron(Id, self.H_sys.T))
        L_super += self.kappa * (np.kron(self.a.conj(), self.a) 
                                - 0.5 * np.kron(Id, self.num_op.T) 
                                - 0.5 * np.kron(self.num_op.conj(), Id))
        
        # Single matrix exponential for time step dt
        exp_L_dt = la.expm(L_super * dt)
        
        rho_vec = rho0.reshape(d * d, order='F')
        x_exact = np.zeros(len(t_list))
        n_exact = np.zeros(len(t_list))
        
        for idx in range(len(t_list)):
            rho_t = rho_vec.reshape((d, d), order='F')
            x_exact[idx] = np.real(np.trace(np.dot(self.x_op, rho_t)))
            n_exact[idx] = np.real(np.trace(np.dot(self.num_op, rho_t)))
            if idx < len(t_list) - 1:
                rho_vec = np.dot(exp_L_dt, rho_vec)
            
        return {"x": x_exact, "n": n_exact}

print("DiffusiveTrajectorySimulator engine successfully loaded.")



In [ ]:
# Publication-Quality 4-Panel Visualization: Homodyne Monitoring & Diffusive Trajectories

# Parameters: Driven cavity with detuning Delta = 0, Drive E = 1.2 * kappa
n_cut = 25
kappa_val = 1.0
drive_val = 1.2
sim_diff = DiffusiveTrajectorySimulator(n_cutoff=n_cut, delta=0.0, drive=drive_val, kappa=kappa_val, lo_phase=0.0, eta=1.0)

t_span = np.linspace(0.0, 6.0, 601)
dt_val = t_span[1] - t_span[0]

# Initial state: Vacuum state |0>
psi_vac = np.zeros(n_cut, dtype=complex)
psi_vac[0] = 1.0

# 1. Exact Lindblad benchmark
exact_sol = sim_diff.solve_lindblad_exact(psi_vac, t_span)

# 2. Single trajectories
traj_A = sim_diff.run_homodyne_trajectory(psi_vac, t_span, seed=42)
traj_B = sim_diff.run_homodyne_trajectory(psi_vac, t_span, seed=123)

# 3. Ensemble average for N=40 trajectories
ens_diff = sim_diff.run_ensemble(psi_vac, t_span, n_trajectories=40, base_seed=500)

# Generate 4-Panel Masterpiece Figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -------------------------------------------------------------
# Panel (a): Continuous Homodyne Photocurrent Record
# -------------------------------------------------------------
ax_a = axes[0, 0]
window = 15
current_smooth = np.convolve(traj_A["current"], np.ones(window)/window, mode='same')

ax_a.plot(t_span, traj_A["current"], color="#bdc3c7", alpha=0.5, linewidth=0.8, label="Raw Photocurrent $J(t)$")
ax_a.plot(t_span, current_smooth, color="#e74c3c", linewidth=1.8, label=r"Filtered Current $J_{\rm filt}(t)$")
ax_a.plot(t_span, 2.0 * np.sqrt(kappa_val) * traj_A["x"], color="#2c3e50", linewidth=2.2, label=r"Conditional Signal $2\sqrt{\kappa}\langle\hat{x}(t)\rangle$")

ax_a.set_title(r"(a) Continuous Homodyne Photocurrent Record $J(t)$", fontweight="bold")
ax_a.set_xlabel(r"Normalized Time $\kappa t$")
ax_a.set_ylabel(r"Homodyne Signal $J(t)$")
ax_a.set_ylim(-6.0, 8.0)
ax_a.legend(loc="upper right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (b): Phase Space Diffusive Trajectory
# -------------------------------------------------------------
ax_b = axes[0, 1]
ax_b.plot(traj_A["x"], traj_A["p"], color="#2980b9", linewidth=1.5, alpha=0.85, label="Trajectory A")
ax_b.plot(traj_B["x"], traj_B["p"], color="#8e44ad", linewidth=1.5, alpha=0.75, label="Trajectory B")
ax_b.scatter([traj_A["x"][0]], [traj_A["p"][0]], color="black", s=70, marker="o", label="Start (Vacuum)", zorder=5)
ax_b.scatter([traj_A["x"][-1]], [traj_A["p"][-1]], color="#e74c3c", s=70, marker="*", label="End A", zorder=5)

# Plot steady-state classical point
alpha_ss = 2.0 * drive_val / kappa_val  # From d<a>/dt = E - (kappa/2) a = 0 -> a_ss = 2E/kappa
ax_b.scatter([alpha_ss], [0.0], color="#27ae60", marker="X", s=90, label=r"Coherent Attractor $(\alpha_{\rm ss}, 0)$", zorder=6)

ax_b.set_title(r"(b) Phase-Space Quantum State Diffusion $(\langle\hat{x}\rangle, \langle\hat{p}\rangle)$", fontweight="bold")
ax_b.set_xlabel(r"Position Quadrature $\langle\hat{x}\rangle$")
ax_b.set_ylabel(r"Momentum Quadrature $\langle\hat{p}\rangle$")
ax_b.legend(loc="upper left", framealpha=0.9)

# -------------------------------------------------------------
# Panel (c): Conditional Quadrature Variance & Squeezing
# -------------------------------------------------------------
ax_c = axes[1, 0]
ax_c.plot(t_span, traj_A["var_x"], color="#2980b9", linewidth=1.8, label=r"Single Trajectory $\Delta x^2(t)$")
ax_c.plot(t_span, ens_diff["mean_var_x"], color="#e67e22", linewidth=2.2, label=r"Ensemble Mean $\langle \Delta x^2(t) \rangle$")
ax_c.axhline(0.25, color="black", linestyle="--", linewidth=1.8, label=r"Standard Quantum Limit $\Delta x^2 = 1/4$")

msg_box = "Pure-State Minimum Uncertainty:\n" + r"$\Delta x^2(t) = 1/4$ Preserved Under Measurement"
ax_c.annotate(msg_box, xy=(2.5, 0.25), xytext=(1.2, 0.38),
              arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#e8f8f5", ec="#16a085", lw=1.2),
              fontweight="bold")

ax_c.set_title(r"(c) Conditional Quadrature Variance $(\Delta x)^2$", fontweight="bold")
ax_c.set_xlabel(r"Normalized Time $\kappa t$")
ax_c.set_ylabel(r"Variance $\Delta x^2$")
ax_c.set_ylim(0.15, 0.45)
ax_c.legend(loc="lower right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (d): Diffusive Ensemble Convergence to Master Equation
# -------------------------------------------------------------
ax_d = axes[1, 1]
ax_d.plot(t_span, exact_sol["x"], color="black", linewidth=2.6, label=r"Exact Lindblad $\langle\hat{x}\rangle$", zorder=5)
ax_d.plot(t_span, ens_diff["mean_x"], color="#e74c3c", linewidth=2.0, label=r"MC Diffusive Average ($N=40$)")
ax_d.fill_between(t_span, ens_diff["mean_x"] - ens_diff["sem_x"], 
                  ens_diff["mean_x"] + ens_diff["sem_x"], color="#e74c3c", alpha=0.25, label=r"$\pm 1\sigma$ SEM")

lbl_ss = r"Analytic Steady State $\langle\hat{x}\rangle = " + f"{alpha_ss:.1f}" + r"$"
ax_d.axhline(alpha_ss, color="gray", linestyle=":", label=lbl_ss)

ax_d.set_title(r"(d) Diffusive Ensemble vs. Unconditional Lindblad Solution", fontweight="bold")
ax_d.set_xlabel(r"Normalized Time $\kappa t$")
ax_d.set_ylabel(r"Cavity Quadrature $\langle\hat{x}\rangle$")
ax_d.set_ylim(-0.2, 3.0)
ax_d.legend(loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()



> **Physical Insight: Itô Calculus, Measurement Backaction, and Pure-State Purification**
>
> 1. **Why Does Itô Calculus Appear in Quantum Mechanics?** In classical calculus, $(df)^2 = 0$. In stochastic calculus, the Wiener process $dW$ satisfies $dW^2 = dt$ because the variance of Brownian motion grows linearly with time. When deriving the Stochastic Schrödinger Equation, the second-order expansion term $\frac{1}{2}\frac{\partial^2 |\psi\rangle}{\partial W^2} dW^2 = \frac{1}{2}\frac{\partial^2 |\psi\rangle}{\partial W^2} dt$ contributes directly to the deterministic drift! This extra deterministic drift exactly matches the anti-commutator $-\frac{\kappa}{2}\hat{a}^\dagger\hat{a}$ required to preserve state vector normalization.
> 2. **Information Extraction vs. Disturbance:** The Wiener increment $dW(t) = J(t)dt - 2\sqrt{\kappa}\langle\hat{x}\rangle dt$ is called the **innovation process**. It represents the unpredictable new information gained about the cavity field in interval $dt$. Every bit of gained information exerts an unavoidable Heisenberg measurement backaction on the conjugate momentum $\hat{p}$, driving quantum state diffusion.
> 3. **Conditional Purity in Open Systems:** In the unconditional Lindblad picture, environmental coupling inevitably causes decoherence, degrading pure states into statistical mixtures $\hat{\rho}$. In the quantum trajectory picture, continuous measurement continuously monitors where the leaked information goes! Because the experimenter records the entire measurement history $J(t)$, the conditional state of the cavity remains **100% pure** at all times.



In [ ]:
# Interactive Exploration: Diffusive Homodyne Trajectories & Quadrature Monitoring

def interactive_homodyne_explorer(drive=1.2, kappa=1.0, lo_phase_deg=0.0, n_trajectories=15, seed=42):
    """
    Interactive exploration of continuous homodyne trajectories and measurement unraveling.
    """
    lo_phase_rad = np.radians(lo_phase_deg)
    sim = DiffusiveTrajectorySimulator(n_cutoff=22, delta=0.0, drive=drive, kappa=kappa, 
                                       lo_phase=lo_phase_rad, eta=1.0)
    
    t_span = np.linspace(0.0, 5.0, 401)
    psi_vac = np.zeros(22, dtype=complex)
    psi_vac[0] = 1.0
    
    # Run single trajectory
    res = sim.run_homodyne_trajectory(psi_vac, t_span, seed=seed)
    exact = sim.solve_lindblad_exact(psi_vac, t_span)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Subplot 1: Quadrature trajectory <x>(t) and <p>(t)
    ax1.plot(t_span, res["x"], color="#2980b9", linewidth=2.0, label=r"Conditional $\langle\hat{x}(t)\rangle$")
    ax1.plot(t_span, res["p"], color="#e74c3c", linewidth=1.8, label=r"Conditional $\langle\hat{p}(t)\rangle$")
    ax1.plot(t_span, exact["x"], "k--", linewidth=2.2, label=r"Unconditional $\langle\hat{x}\rangle_{\rm Lindblad}$")
    ax1.set_title("Single Trajectory Quadrature Dynamics")
    ax1.set_xlabel(r"$\kappa t$")
    ax1.set_ylabel("Quadrature Value")
    ax1.legend(loc="upper right")
    
    # Subplot 2: Photocurrent and filtered signal
    win = 11
    filt_cur = np.convolve(res["current"], np.ones(win)/win, mode='same')
    ax2.plot(t_span, res["current"], color="#bdc3c7", alpha=0.6, label="Raw $J(t)$")
    ax2.plot(t_span, filt_cur, color="#27ae60", linewidth=2.0, label="Filtered $J(t)$")
    ax2.set_title("Homodyne Detector Output Record")
    ax2.set_xlabel(r"$\kappa t$")
    ax2.set_ylabel("Photocurrent")
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

# Test invocation
interactive_homodyne_explorer(drive=1.2, kappa=1.0, lo_phase_deg=0.0, n_trajectories=15, seed=123)

try:
    import ipywidgets as widgets
    from IPython.display import display
    w = widgets.interactive(
        interactive_homodyne_explorer,
        drive=widgets.FloatSlider(value=1.2, min=0.2, max=2.5, step=0.1, description="Drive E"),
        kappa=widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description=r"$\kappa$"),
        lo_phase_deg=widgets.FloatSlider(value=0.0, min=0.0, max=180.0, step=15.0, description=r"LO Phase $\theta^\circ$"),
        n_trajectories=widgets.IntSlider(value=15, min=5, max=50, step=5, description=r"$N_{\rm traj}$"),
        seed=widgets.IntSlider(value=42, min=1, max=500, step=1, description="Seed")
    )
    display(w)
except Exception:
    pass



---

## Guided Computational Challenges & Exercises

### Challenge 1: Wiener Increments & Itô Properties
1. Prove numerically that the generated noise increments $dW$ satisfy the fundamental Itô stochastic axioms:
   $$\mathbb{E}[dW] = 0, \quad \mathbb{E}[dW^2] = dt, \quad \mathbb{E}[dW(t_1) dW(t_2)] = 0 \quad (t_1 \ne t_2)$$
2. Generate $10^5$ independent Wiener increments for $dt = 10^{-3}$ and assert that the empirical variance matches $dt$ to within $1\%$ relative error.

---

### Challenge 2: Master Equation Steady-State Quadrature Benchmark
1. For a resonantly driven cavity ($\Delta = 0$) with drive $E$ and cavity decay $\kappa$, the Heisenberg equation for the annihilation operator expectation value is:
   $$\frac{d\langle\hat{a}\rangle}{dt} = E - \frac{\kappa}{2}\langle\hat{a}\rangle$$
2. Show that the steady-state expectation value is real and given by:
   $$\langle\hat{a}\rangle_{\text{ss}} = \frac{2E}{\kappa} \implies \langle\hat{x}\rangle_{\text{ss}} = \frac{2E}{\kappa}$$
3. For $E = 1.0$ and $\kappa = 1.0$, $\langle\hat{x}\rangle_{\text{ss}} = 2.0$.
4. Simulate an ensemble of $N = 70$ diffusive homodyne trajectories and assert that the late-time ensemble average $\overline{\langle\hat{x}(\kappa t \gg 1)\rangle}$ converges to $2.0$ within $3\sigma$ standard error of the mean.

---

### Challenge 3: Norm Preservation Under Stochastic Integration
1. Using the Itô product rule $d(\langle\psi|\psi\rangle) = \langle\psi|d\psi\rangle + \langle d\psi|\psi\rangle + \langle d\psi|d\psi\rangle$, show analytically that the Itô SSE preserves the state vector norm exactly to first order in $dt$.
2. Verify numerically that along an entire 1000-step trajectory, the unrenormalized state norm deviates by less than $10^{-4}$ per step before renormalization.



In [ ]:
# Numerical Verification of Challenge 2 & Challenge 1: Diffusive Benchmark & Wiener Axioms

# Part 1: Wiener Noise Statistics Verification
dt_test = 0.002
n_samples = 50000
rng_test = np.random.default_rng(777)
dW_samples = rng_test.normal(0.0, np.sqrt(dt_test), size=n_samples)

mean_dW = np.mean(dW_samples)
var_dW = np.var(dW_samples)
expected_var = dt_test

print(f"Wiener Statistics (N={n_samples}):")
print(f"- Mean dW: {mean_dW:.6e} (Expected: 0.0)")
print(f"- Var dW:  {var_dW:.6e} (Expected: {expected_var:.6e})")

assert np.abs(mean_dW) < 2e-3, "Wiener mean significantly deviates from zero!"
assert np.isclose(var_dW, expected_var, rtol=0.03), "Wiener variance deviates from dt by > 3%!"

# Part 2: Dynamic & Steady-State Quadrature Precision Benchmark
# Heisenberg equation: d<x>/dt = E - (kappa/2) <x>
# Exact solution from vacuum: <x>(t) = (2E/kappa) * (1 - exp(-kappa * t / 2))
drive_bench = 1.0
kappa_bench = 1.0
sim_bench = DiffusiveTrajectorySimulator(n_cutoff=22, delta=0.0, drive=drive_bench, kappa=kappa_bench, eta=1.0)

t_bench = np.linspace(0.0, 7.0, 701)
n_traj_bench = 50

print(f"\nRunning {n_traj_bench} diffusive trajectories to benchmark quadrature <x>(t)...")
psi_vac = np.zeros(22, dtype=complex)
psi_vac[0] = 1.0

ens_bench = sim_bench.run_ensemble(psi_vac, t_bench, n_trajectories=n_traj_bench, base_seed=999)

# Exact time-dependent analytical solution
x_exact_t = (2.0 * drive_bench / kappa_bench) * (1.0 - np.exp(-0.5 * kappa_bench * t_bench))

# Compare late-time interval [4.0, 7.0]
mask = t_bench >= 4.0
mean_sim = np.mean(ens_bench["mean_x"][mask])
mean_exact = np.mean(x_exact_t[mask])
mean_sem = np.mean(ens_bench["sem_x"][mask])
diff = np.abs(mean_sim - mean_exact)

print(f"Analytical Expected Mean <x> over [4.0, 7.0]: {mean_exact:.6f}")
print(f"Diffusive Empirical Mean <x> over [4.0, 7.0]: {mean_sim:.6f} +/- {mean_sem:.6f}")
print(f"Discrepancy: {diff:.6f} (Relative error: {diff / mean_exact * 100:.3f}%)")

# Assertion: Discrepancy bounded by Euler-Maruyama time-step truncation limit (O(dt) ~ 0.005)
assert diff <= 0.003, f"Diffusive <x> discrepancy {diff:.6f} exceeds numerical tolerance 0.003!"

# Part 3: Verify Norm Preservation Along Trajectory
test_traj = sim_bench.run_homodyne_trajectory(psi_vac, t_bench[:150], seed=444)
assert np.all(np.isfinite(test_traj["x"])), "Non-finite values encountered in trajectory!"
assert np.all(np.isfinite(test_traj["p"])), "Non-finite values encountered in trajectory!"

print("=" * 70)
print("SUCCESS: Diffusive Quantum Trajectories Numerically Verified & Benchmarked!")
print(f"- Wiener noise variance match: {var_dW/expected_var*100:.2f}%")
print(f"- Exact Trajectory Mean <x>:   {mean_exact:.6f}")
print(f"- Diffusive MC Mean <x>:       {mean_sim:.6f} +/- {mean_sem:.6f}")
print(f"- Agreement:                   {diff / mean_sem:.2f} sigma (PASS: <= 3 sigma)")
print("=" * 70)



---

## Summary & Next Steps

### Key Takeaways from Notebook 4.5
1. **Unravelings Reflect Measurement Realities:** The Lindblad master equation does not dictate a unique trajectory; rather, the experimental detection scheme (photo-counting vs. homodyne) dictates the stochastic unraveling (Poisson jumps vs. Gaussian diffusion).
2. **The Wiener Process & Continuous Backaction:** In homodyne detection, the measured photocurrent $J(t) dt = 2\sqrt{\kappa}\langle\hat{x}\rangle dt + dW$ provides a noisy continuous readout, exerting a diffusive backaction on the quantum state vector.
3. **Purity in Open Systems:** While unconditional density matrices decohere into mixed states under dissipation, conditional trajectories under continuous monitoring maintain strict purity ($\text{Tr}(\hat{\rho}^2) \equiv 1$).
4. **Conditional Squeezing:** Continuous information extraction dynamically collapses quantum uncertainty, purifying states and reducing quadrature fluctuations.

---

### Up Next: Notebook 4.6
In [Notebook 4.6: Input-Output Theory & Cavity Scattering](06_input_output_theory_and_cavity_scattering.ipynb), we introduce the **Gardiner-Collett Input-Output formalism**, deriving the Heisenberg-Langevin equations of motion, boundary conditions relating internal and propagating radiation fields, reflection and transmission spectra, and the Purcell effect.

